In [ ]:
# =====================================================
# STEP 0: IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, log_loss, confusion_matrix

import warnings
warnings.filterwarnings("ignore")


# =====================================================
# STEP 1: LOAD DATA (CHANGE PATH ONLY)
# =====================================================
train = pd.read_csv("/kaggle/input/mock-test-2-mse-2/train.csv")
test  = pd.read_csv("/kaggle/input/mock-test-2-mse-2/test.csv")

print("Train:", train.shape)
print("Test :", test.shape)


# =====================================================
# STEP 2: 🔥 UNIVERSAL CONFIGURATION (ONLY CHANGE HERE)
# =====================================================
PROBLEM_TYPE = "multiclass"
# "binary"     → Q1 (Objects), Q3 (Obesity)
# "multiclass" → Q2 (Log Loss)

TARGET_COLUMN = "Status"
# Q1 → "Class"
# Q2 → "Status"
# Q3 → "NObeyesdad"

SUBMISSION_TYPE = "probability"
# Q1/Q3 → "label"
# Q2    → "probability"

ID_COLUMN = "id"


# =====================================================
# STEP 3: BASIC EDA
# =====================================================
print("\n🔹 Missing Values\n", train.isnull().sum())
print("\n🔹 Target Distribution\n", train[TARGET_COLUMN].value_counts())

plt.figure(figsize=(6,6))
train[TARGET_COLUMN].value_counts().plot.pie(autopct="%1.1f%%")
plt.title("Target Distribution")
plt.ylabel("")
plt.show()


# =====================================================
# STEP 4: SPLIT FEATURES & TARGET
# =====================================================
X = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

if ID_COLUMN in X.columns:
    X = X.drop(columns=[ID_COLUMN])
    test_features = test.drop(columns=[ID_COLUMN])
else:
    test_features = test.copy()


# =====================================================
# STEP 5: DATA CLEANING
# =====================================================
for col in X.columns:
    if X[col].dtype == "object":
        X[col].fillna(X[col].mode()[0], inplace=True)
        test_features[col].fillna(test_features[col].mode()[0], inplace=True)
    else:
        X[col].fillna(X[col].median(), inplace=True)
        test_features[col].fillna(test_features[col].median(), inplace=True)


# =====================================================
# STEP 6: HISTOGRAM + BOXPLOT
# =====================================================
X.hist(figsize=(15,10))
plt.show()

plt.figure(figsize=(12,5))
sns.boxplot(data=X.select_dtypes(include=np.number))
plt.xticks(rotation=90)
plt.show()


# =====================================================
# STEP 7: ENCODING (SAFE)
# =====================================================
cat_cols = X.select_dtypes(include="object").columns

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])
test_features[cat_cols] = encoder.transform(test_features[cat_cols])

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)


# =====================================================
# STEP 8: FEATURE SCALING
# =====================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)
test_scaled = scaler.transform(test_features[X.columns].values)


# =====================================================
# STEP 9: CORRELATION
# =====================================================
plt.figure(figsize=(10,7))
sns.heatmap(pd.DataFrame(X_scaled).corr(), cmap="coolwarm")
plt.show()


# =====================================================
# STEP 10: TRAIN–VALIDATION SPLIT
# =====================================================
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


# =====================================================
# STEP 11: MODEL
# =====================================================
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)


# =====================================================
# STEP 12: EVALUATION
# =====================================================
y_pred = rf.predict(X_val)
y_prob = rf.predict_proba(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))

if PROBLEM_TYPE == "multiclass":
    print("Log Loss:", log_loss(y_val, y_prob))

print(classification_report(y_val, y_pred))

cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.show()


# =====================================================
# STEP 13: HYPERPARAMETER TUNING
# =====================================================
scoring_metric = "neg_log_loss" if PROBLEM_TYPE == "multiclass" else "accuracy"

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators":[100,200],
        "max_depth":[None,10,20]
    },
    cv=3,
    scoring=scoring_metric,
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_


# =====================================================
# STEP 14: FINAL TRAINING
# =====================================================
best_model.fit(X_scaled, y)


# =====================================================
# STEP 15: SUBMISSION
# =====================================================
if SUBMISSION_TYPE == "probability":
    probs = best_model.predict_proba(test_scaled)
    submission = pd.DataFrame(
        probs,
        columns=[f"{TARGET_COLUMN}_{c}" for c in target_encoder.classes_]
    )
else:
    preds = best_model.predict(test_scaled)
    preds = target_encoder.inverse_transform(preds)
    submission = pd.DataFrame({TARGET_COLUMN: preds})

submission.insert(0, ID_COLUMN, test[ID_COLUMN])
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv READY")